Q. Bayesian Estimation of a User Ability Parameter from Item Responses

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =====================================================================
# TASK 1: VISUALIZING THE MECHANICS (2PL IRT CURVES)
# =====================================================================
print("Generating Plot 1: IRT Curves...")
theta_domain = np.linspace(-4, 4, 500)

def p_correct(t, a, b):
    return 1.0 / (1.0 + np.exp(-a * (t - b)))

# Configurations: One 'a' value with three 'b' values, plus a unique 'a' value
configs = [
    {"a": 1.5, "b": -1.0, "name": "High Discrim, Low Difficulty (a=1.5, b=-1)", "color": "blue"},
    {"a": 1.5, "b": 0.0,  "name": "High Discrim, Mid Difficulty (a=1.5, b=0)",  "color": "cyan"},
    {"a": 1.5, "b": 1.5,  "name": "High Discrim, High Difficulty (a=1.5, b=1.5)", "color": "darkblue"},
    {"a": 0.5, "b": 0.0,  "name": "Low Discrim, Mid Difficulty (a=0.5, b=0)",   "color": "crimson"}
]

fig1 = go.Figure()
for c in configs:
    fig1.add_trace(go.Scatter(
        x=theta_domain, y=p_correct(theta_domain, c["a"], c["b"]),
        mode='lines', name=c["name"], line=dict(color=c["color"], width=2.5)
    ))

fig1.update_layout(
    title="Task 1: 2PL IRT Curve Mechanics (Discrimination vs Difficulty)",
    xaxis_title="Latent User Ability (θ)",
    yaxis_title="P(Y=1|θ)",
    template="plotly_white"
)
fig1.show()


# =====================================================================
# TASK 6 & 7: NUMERICAL GRID SIMULATION & TIMELINE CONVERGENCE
# =====================================================================
print("Running Timeline Simulation (Tasks 6 & 7)...")
np.random.seed(42)  # For reproducibility
theta_true = 0.75
n_items = 20

# Generate items dynamically[cite: 1]
b_items = np.random.normal(0, 1, n_items)
a_items = np.random.uniform(0.5, 2.0, n_items)

# Simulate user responses[cite: 1]
true_probs = p_correct(theta_true, a_items, b_items)
responses = (np.random.uniform(0, 1, n_items) < true_probs).astype(int)

# Setup numerical grid[cite: 1]
grid_size = 1000
theta_grid = np.linspace(-4, 4, grid_size)

# Initialize prior: Standard Normal N(0, 1)[cite: 1]
prior_init = np.exp(-theta_grid**2 / 2.0) / np.sqrt(2 * np.pi)
posterior = prior_init.copy()

# Trajectory tracking lists[cite: 1]
bayes_estimates = [0.0]
map_estimates = [0.0]
steps = list(range(n_items + 1))

# Isolated check variable for Task 6 (we will grab the first update state to show)
first_update_posterior = None
task6_normalization_constant = 0.0

# Sequential filtering loop[cite: 1]
for k in range(n_items):
    y_k = responses[k]
    a_k = a_items[k]
    b_k = b_items[k]

    # Calculate likelihood contribution[cite: 1]
    p_k = p_correct(theta_grid, a_k, b_k)
    likelihood = p_k if y_k == 1 else (1.0 - p_k)

    # Unnormalized update[cite: 1]
    unnormalized = posterior * likelihood

    # Normalization step using the trapezoidal rule[cite: 1]
    integral = np.trapezoid(unnormalized, theta_grid)
    if integral > 0:
        posterior = unnormalized / integral

    # Extract isolated first step data for Task 6 demonstration
    if k == 0:
        first_update_posterior = posterior.copy()
        task6_normalization_constant = integral

    # Calculate Point Estimators via numerical integration[cite: 1]
    theta_bayes = np.trapezoid(posterior * theta_grid, theta_grid)
    theta_map = theta_grid[np.argmax(posterior)]

    bayes_estimates.append(theta_bayes)
    map_estimates.append(theta_map)

# ---------------------------------------------------------------------
# Plot 2: Task 6 Single-Step Grid Update Verification
# ---------------------------------------------------------------------
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=theta_grid, y=prior_init, mode='lines', name='Initial Prior Step 0', line=dict(dash='dash', color='gray')))
fig2.add_trace(go.Scatter(x=theta_grid, y=first_update_posterior, mode='lines', name='Posterior Step 1', line=dict(color='green', width=2.5)))
fig2.update_layout(
    title=f"Task 6: Algorithmic Verification of a Single Grid Update (Norm Constant: {task6_normalization_constant:.4f})",
    xaxis_title="Ability Coordinate (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)
fig2.show()

# ---------------------------------------------------------------------
# Plot 3: Task 7 Dynamic Timeline Estimator Convergence
# ---------------------------------------------------------------------
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=steps, y=[theta_true] * len(steps), mode='lines', name='True Ability (θ = 0.75)', line=dict(color='black', dash='dash', width=2)))
fig3.add_trace(go.Scatter(x=steps, y=bayes_estimates, mode='lines+markers', name='Posterior Mean (θ_Bayes)', line=dict(color='blue', width=2.5)))
fig3.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='Maximum A Posteriori (θ_MAP)', line=dict(color='crimson', width=2)))

fig3.update_layout(
    title="Task 7: Sequential Convergence Tracking Timeline",
    xaxis_title="Item Timeline Step (k)",
    yaxis_title="Latent Ability Estimate (θ)",
    template="plotly_white",
    hovermode="x unified"
)
fig3.show()

Generating Plot 1: IRT Curves...


Running Timeline Simulation (Tasks 6 & 7)...


Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# TASK 1: STRUCTURAL PROBABILITY AND PROPERTIES (BETA DENSITIES)
# =====================================================================
print("Generating Plot 1: Beta Domain Distributions...")
theta_domain = np.linspace(0, 1, 500)

# Continuous shape parameter pair arrays
beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative state: Beta(1,1)", "color": "gray"},
    {"alpha": 2, "beta": 8, "name": "Right-skewed state: Beta(2,8)", "color": "crimson"},
    {"alpha": 8, "beta": 2, "name": "Left-skewed state: Beta(8,2)", "color": "darkcyan"}
]

fig1 = go.Figure()
for config in beta_configs:
    pdf_vals = stats.beta.pdf(theta_domain, config["alpha"], config["beta"])
    fig1.add_trace(go.Scatter(
        x=theta_domain, y=pdf_vals, mode='lines',
        name=config["name"], line=dict(color=config["color"], width=2.5)
    ))

fig1.update_layout(
    title="Task 1: Prior Probability Density Functions for Beta Structural States",
    xaxis_title="Click-Through Rate (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)
fig1.show()


# =====================================================================
# TASK 6: PERFORMANCE TRACKING AND CONVERGENCE SIMULATION
# =====================================================================
print("Running Timeline Simulation (Tasks 3, 5, & 6)...")
np.random.seed(101)  # For reproducibility
theta_true = 0.35
n_impressions = 100

# Initialize State: Base uniform prior parameters[cite: 1]
alpha_param = 1.0
beta_param = 1.0

# Dynamically generate binary user interactions (1 = Click, 0 = Non-click)[cite: 1]
clicks = (np.random.uniform(0, 1, n_impressions) < theta_true).astype(int)

# Storage track registers (Step 0 holds initial prior status)[cite: 1]
bayes_track = [alpha_param / (alpha_param + beta_param)]
map_track = [0.5]  # Standardized center point assignment for structural uniform state
steps = list(range(n_impressions + 1))

# Sequential Closed-Form Conjugate Loop[cite: 1]
for k in range(n_impressions):
    y_k = clicks[k]

    # Task 3: Closed-Form Analytical Arithmetic Updates[cite: 1]
    alpha_param += y_k
    beta_param += (1 - y_k)

    # Task 5: Exact Equations for Evaluated Point Estimators[cite: 1]
    # Running Posterior Mean: E[θ] = α / (α + β)
    theta_bayes = alpha_param / (alpha_param + beta_param)

    # Running Maximum A Posteriori (MAP): Mode = (α - 1) / (α + β - 2)
    if alpha_param > 1 and beta_param > 1:
        theta_map = (alpha_param - 1) / (alpha_param + beta_param - 2)
    else:
        theta_map = theta_bayes  # Mode fallback context for uniform boundaries

    bayes_track.append(theta_bayes)
    map_track.append(theta_map)

# ---------------------------------------------------------------------
# Plot 2: Dynamic Timeline Estimator Convergence Tracker
# ---------------------------------------------------------------------
fig2 = go.Figure()

# True CTR horizontal reference line[cite: 1]
fig2.add_trace(go.Scatter(
    x=steps, y=[theta_true] * len(steps),
    mode='lines', name='True CTR (θ_true = 0.35)',
    line=dict(color='black', dash='dot', width=2)
))

# Running Posterior Mean[cite: 1]
fig2.add_trace(go.Scatter(
    x=steps, y=bayes_track,
    mode='lines', name='Running Posterior Mean (θ_Bayes)',
    line=dict(color='darkcyan', width=2.5)
))

# Running MAP[cite: 1]
fig2.add_trace(go.Scatter(
    x=steps, y=map_track,
    mode='lines', name='Running MAP (θ_MAP)',
    line=dict(color='orange', width=2)
))

fig2.update_layout(
    title="Task 6: Sequential CTR Estimator Convergence Over Timeline",
    xaxis_title="Impression Timeline Milestones (k)",
    yaxis_title="Estimated Conversion Rate (θ)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)
fig2.show()

Generating Plot 1: Beta Domain Distributions...


Running Timeline Simulation (Tasks 3, 5, & 6)...


Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# TASK 1: PRIOR BELIEF BOUNDARIES VISUALIZATION
# =====================================================================
print("Generating Plot 1: Bounded Prior Density...")
grid_size = 500
theta_grid = np.linspace(0.01, 1.0, grid_size)

# Bounded Prior Distribution: Beta(8, 1.5)[cite: 1]
prior_pdf = stats.beta.pdf(theta_grid, 8, 1.5)

# Calculate analytical expectation: E[Θ] = α / (α + β)
expected_prior = 8 / (8 + 1.5)
print(f"Analytical Expected Prior Stiffness Efficiency: {expected_prior:.4f}")

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=theta_grid, y=prior_pdf, mode='lines',
    name='Initial Prior Beta(8, 1.5)', line=dict(color='indigo', width=2.5)
))
fig1.add_trace(go.Scatter(
    x=[expected_prior], y=[0], mode='markers+text',
    text=[f" E[Θ] = {expected_prior:.3f}"], textposition="top right",
    marker=dict(color='crimson', size=10, symbol='diamond'),
    name='Prior Expectation'
))

fig1.update_layout(
    title="Task 1: Initial Optimistic Prior Density Over Physical Domain",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)
fig1.show()


# =====================================================================
# TASK 5 & 6: SIMULATION, GRID CONFIGURATIONS & CONVERGENCE TIMELINE
# =====================================================================
print("Running Timeline Simulation (Tasks 5 & 6)...")
np.random.seed(7)  # For reproducible log-normal noise tracking
theta_true = 0.68
n_inspections = 15
K_nominal = 50.0
sigma = 0.15

# Simulate Sensor Stream via Multiplicative Log-Normal Noise[cite: 1]
epsilons = np.random.normal(0, sigma, n_inspections)
y_stream = theta_true * K_nominal * np.exp(epsilons)

# Initialize grid variables dynamically using the baseline prior state
posterior = stats.beta.pdf(theta_grid, 8, 1.5)
posterior /= np.trapezoid(posterior, theta_grid)  # Initial normalization config

# Setup logging records
bayes_history = [np.trapezoid(posterior * theta_grid, theta_grid)]
map_history = [theta_grid[np.argmax(posterior)]]
milestones = {0: posterior.copy()}

# Sequential Algorithmic Non-Conjugate Grid Update Loop
for k in range(1, n_inspections + 1):
    y_k = y_stream[k-1]

    # Task 2/3: Log-Normal Structural Likelihood Equation Formulation[cite: 1]
    log_diff = np.log(y_k / (theta_grid * K_nominal))
    likelihood = (1.0 / (y_k * sigma * np.sqrt(2 * np.pi))) * np.exp(- (log_diff)**2 / (2 * sigma**2))

    # Recursive Multiplicative Update Rule[cite: 1]
    posterior = posterior * likelihood

    # Task 5: Sequential Normalization Using the Trapezoidal Rule[cite: 1]
    area = np.trapezoid(posterior, theta_grid)
    if area > 0:
        posterior /= area

    # Task 4/5: Running Definite Integral Points Evaluation[cite: 1]
    theta_bayes = np.trapezoid(posterior * theta_grid, theta_grid)
    theta_map = theta_grid[np.argmax(posterior)]

    bayes_history.append(theta_bayes)
    map_history.append(theta_map)

    # Capture requested timeline milestones[cite: 1]
    if k in [1, 2, 5, 10, 15]:
        milestones[k] = posterior.copy()

# ---------------------------------------------------------------------
# Plot 2: Task 6.1 Progression of Full Posterior Density Curves
# ---------------------------------------------------------------------
fig2 = go.Figure()
colors = ['#cccccc', '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for idx, (step_k, pdf_vals) in enumerate(milestones.items()):
    fig2.add_trace(go.Scatter(
        x=theta_grid, y=pdf_vals, mode='lines',
        name=f'Milestone k = {step_k}',
        line=dict(width=2.5 if step_k in [0, 15] else 1.8, color=colors[idx])
    ))

fig2.update_layout(
    title="Task 6.1: Evolution of Structural Posterior Density Curves Across Milestones",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)
fig2.show()

# ---------------------------------------------------------------------
# Plot 3: Task 6.2 Convergence Tracker Timeline
# ---------------------------------------------------------------------
fig3 = go.Figure()
steps = list(range(n_inspections + 1))

# True stiffness structural limit boundary line[cite: 1]
fig3.add_trace(go.Scatter(
    x=steps, y=[theta_true] * len(steps), mode='lines',
    name='True Degradation State (θ = 0.68)',
    line=dict(color='black', dash='dash', width=2)
))

# Posterior Mean profile curve[cite: 1]
fig3.add_trace(go.Scatter(
    x=steps, y=bayes_history, mode='lines+markers',
    name='Running Posterior Mean (θ_Bayes)',
    line=dict(color='purple', width=2)
))

# MAP tracking profile curve[cite: 1]
fig3.add_trace(go.Scatter(
    x=steps, y=map_history, mode='lines+markers',
    name='Running MAP Estimate (θ_MAP)',
    line=dict(color='forestgreen', width=2)
))

fig3.update_layout(
    title="Task 6.2: Structural Health Monitoring Estimator Convergence Timeline",
    xaxis_title="Inspection Milestone Step (k)",
    yaxis_title="Stiffness Efficiency Factor (θ)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.01, xanchor="left", x=0.01)
)
fig3.show()

Generating Plot 1: Bounded Prior Density...
Analytical Expected Prior Stiffness Efficiency: 0.8421


Running Timeline Simulation (Tasks 5 & 6)...


Q. Gaussian Mixture Clustering as Conditional Updating

In [4]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
import plotly.express as px

# =====================================================================
# DATA GENERATION (Simulating the Kaggle CC Data Structure)
# =====================================================================
def generate_mock_financial_data(n_samples=1000):
    """Generates synthetic multimodal financial behaviors matching the assignment criteria."""
    np.random.seed(42)
    # Cluster 1: Low purchases, moderate credit limit
    c1 = np.random.multivariate_normal([1500, 3000], [[400**2, 0.2*400*800], [0.2*400*800, 800**2]], int(n_samples*0.4))
    # Cluster 2: High purchases, high credit limit
    c2 = np.random.multivariate_normal([6000, 9000], [[1500**2, 0.6*1500*2000], [0.6*1500*2000, 2000**2]], int(n_samples*0.35))
    # Cluster 3: Moderate purchases, low credit limit
    c3 = np.random.multivariate_normal([3500, 1500], [[600**2, -0.1*600*400], [-0.1*600*400, 400**2]], int(n_samples*0.25))

    data = np.vstack([c1, c2, c3])
    return pd.DataFrame(data, columns=['PURCHASES', 'CREDIT_LIMIT'])


# =====================================================================
# TASK 10: GMM FINANCIAL SEGMENTER IMPLEMENTATION
# =====================================================================
class GMMFinancialSegmenter:
    def __init__(self, n_components=3):
        self.n_components = n_components
        self.scaler = StandardScaler()
        self.model = GaussianMixture(n_components=n_components, random_state=42)

    def prepare_data(self, df):
        """Standardizes features and splits data into an 80/20 train/test profile."""
        X = df[['PURCHASES', 'CREDIT_LIMIT']].values
        X_scaled = self.scaler.fit_transform(X)

        # 80% training set and a 20% validation/test set split
        self.X_train, self.X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)
        return self.X_train, self.X_test

    def fit(self):
        """Fits the GMM via the EM algorithm and monitors tracking convergence metrics."""
        self.model.fit(self.X_train)
        print(f"--- EM Execution Status ---")
        print(f"Model Successfully Converged: {self.model.converged_}")
        print(f"Iterations Required: {self.model.n_iter_}\n")

    def evaluate(self):
        """Computes average out-of-sample log-likelihood score metrics."""
        avg_log_likelihood = self.model.score(self.X_test)
        print(f"--- Out-of-Sample Performance ---")
        print(f"Average Log-Likelihood Score: {avg_log_likelihood:.4f}\n")
        return avg_log_likelihood

    def plot_density_heatmap(self):
        """Figure 1: 2D Density Heatmap of raw training inputs with marginal components."""
        df_train = pd.DataFrame(self.X_train, columns=['PURCHASES', 'CREDIT_LIMIT'])
        fig = px.density_heatmap(
            df_train, x='PURCHASES', y='CREDIT_LIMIT',
            marginal_x="histogram", marginal_y="histogram",
            title="Figure 1: Empirical 2D Density Heatmap of Training Data with Marginals",
            labels={'PURCHASES': 'Standardized Purchases', 'CREDIT_LIMIT': 'Standardized Credit Limit'}
        )
        fig.update_layout(template='plotly_white')
        fig.show()

    def _plot_assignments(self, data_points, title_text):
        """Internal plotting framework to overlay points on continuous posterior field grids."""
        # 1. Coordinate Grid Setup Bounds
        x_min, x_max = self.X_train[:, 0].min() - 0.5, self.X_train[:, 0].max() + 0.5
        y_min, y_max = self.X_train[:, 1].min() - 0.5, self.X_train[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        # 2. Extract Max Soft Responsibility field mapping structures (γ_ik)
        responsibilities = self.model.predict_proba(grid_points)
        max_resp = np.max(responsibilities, axis=1).reshape(xx.shape)

        # 3. Determine hard categorical labels for observation layouts
        hard_labels = self.model.predict(data_points)

        fig = go.Figure()

        # Draw underlay mapping tracking soft confidence assignments
        fig.add_trace(go.Contour(
            x=np.linspace(x_min, x_max, 200),
            y=np.linspace(y_min, y_max, 200),
            z=max_resp, colorscale='Viridis', opacity=0.7,
            colorbar=dict(title="Max Responsibility max_k(γ_ik)"),
            showscale=True
        ))

        # Overplot observation markers grouped by assignment allocations
        fig.add_trace(go.Scatter(
            x=data_points[:, 0], y=data_points[:, 1],
            mode='markers',
            marker=dict(
                color=hard_labels,
                colorscale='Electric',
                size=5,
                line=dict(width=0.5, color='white')
            ),
            name='Assigned Components'
        ))

        fig.update_layout(
            title=title_text,
            xaxis_title='Standardized Purchases',
            yaxis_title='Standardized Credit Limit',
            template='plotly_white'
        )
        fig.show()

    def plot_training_assignments(self):
        """Figure 2: Training data points over maximum posterior responsibilities contour fields."""
        self._plot_assignments(self.X_train, "Figure 2: Training Assignment Plot Over Continuous Posterior Responsibilities")

    def plot_test_assignments(self):
        """Figure 3: Out-of-sample validation data points over assignment boundaries maps."""
        self._plot_assignments(self.X_test, "Figure 3: Test Assignment Boundary Validation Plot (Out-of-Sample)")


# =====================================================================
# EXECUTION PIPELINE
# =====================================================================
if __name__ == "__main__":
    # Load environment mock payload tracking setup
    df_cc = generate_mock_financial_data()

    # Process pipeline model operations
    segmenter = GMMFinancialSegmenter(n_components=3)
    segmenter.prepare_data(df_cc)
    segmenter.fit()
    segmenter.evaluate()

    # Generate Interactive Graph Outputs
    segmenter.plot_density_heatmap()
    segmenter.plot_training_assignments()
    segmenter.plot_test_assignments()

--- EM Execution Status ---
Model Successfully Converged: True
Iterations Required: 5

--- Out-of-Sample Performance ---
Average Log-Likelihood Score: -1.7356

